# McLaren Race Intelligence Platform — Data Update

This notebook keeps the lab's F1 dataset current. It finds the latest round already
in GCS, fetches any newer rounds from the Jolpica API, and overwrites the affected
Parquet files. Students loading data in the lab will automatically get the latest.

**Run this notebook before each lab event to freshen the data.**

Tables updated:
- `races`, `results`, `qualifying`, `sprint_results`
- `driver_standings`, `constructor_standings`
- `drivers`, `constructors` (new entrants only)

Tables not updated (static or too large):
- `circuits`, `seasons`, `status` — effectively static
- `lap_times`, `pit_stops` — too large, not core to lab objectives
- `constructor_results` — derivable from results; rarely queried directly

**Requirements:** Colab Enterprise (IAM auth automatic), write access to `gs://class-demo`

---
## Section 0: Setup

In [ ]:
!pip install -q google-cloud-storage pandas pyarrow requests

In [ ]:
import google.auth
import pandas as pd
import numpy as np
import requests
import time
import re
import io
from google.cloud import storage

credentials, PROJECT_ID = google.auth.default()

# ── Constants ────────────────────────────────────────────────────────────────
GCS_BUCKET       = "class-demo"
GCS_PREFIX       = "mclaren-f1"
GCS_BQ_STAGING   = f"{GCS_PREFIX}/bq-staging"
MCLAREN_REF      = "mclaren"
JOLPICA_BASE     = "https://api.jolpi.ca/ergast/f1"
RATE_SLEEP       = 3.0    # seconds between requests
RETRY_BACKOFF    = 30.0   # seconds to wait on 429
MAX_RETRIES      = 3

gcs_client = storage.Client(project=PROJECT_ID)

print(f"✅ Project  : {PROJECT_ID}")
print(f"✅ GCS      : gs://{GCS_BUCKET}/{GCS_BQ_STAGING}/")

---
## Section 1: Load Current Parquet Files from GCS

Read all affected tables into memory so we can find the latest round
and later merge and overwrite with updated data.

In [ ]:
# Load Parquet files from GCS into a tables dict.
# We read only the tables we intend to update.

TABLES_TO_UPDATE = [
    "races", "results", "qualifying", "sprint_results",
    "driver_standings", "constructor_standings",
    "drivers", "constructors",
]

# These tables are carried through unchanged (for reference/dedup key lookups)
TABLES_STATIC = ["circuits", "seasons", "status", "constructor_results",
                 "lap_times", "pit_stops"]

bucket = gcs_client.bucket(GCS_BUCKET)
tables = {}

print("Loading tables from GCS...\n")
for name in TABLES_TO_UPDATE:
    blob_path = f"{GCS_BQ_STAGING}/{name}.parquet"
    blob = bucket.blob(blob_path)
    if blob.exists():
        data = blob.download_as_bytes()
        tables[name] = pd.read_parquet(io.BytesIO(data))
        print(f"  ✅ {name:<30} {len(tables[name]):>8,} rows")
    else:
        print(f"  ❌ {name:<30} NOT FOUND — gs://{GCS_BUCKET}/{blob_path}")

missing = [t for t in TABLES_TO_UPDATE if t not in tables]
if missing:
    raise RuntimeError(f"Cannot continue — missing tables: {missing}")

print(f"\n✅ All {len(tables)} tables loaded")

---
## Section 2: Find Latest Round and Determine What to Fetch

In [ ]:
# Find the latest year and round already in the data.
# We use the results table as the source of truth — a round only counts
# as 'present' if we have actual race results for it, not just a scheduled entry.

results_df = tables["results"]
races_df   = tables["races"]

# Join results with races to get year and round
results_with_round = results_df.merge(
    races_df[["race_id", "year", "round"]], on="race_id", how="left"
)

latest_year  = int(results_with_round["year"].max())
latest_round = int(
    results_with_round[results_with_round["year"] == latest_year]["round"].max()
)

print(f"Latest data in GCS : {latest_year} Round {latest_round}")

# Ask Jolpica for the current season's race schedule to find what's available
import datetime
current_year = datetime.date.today().year
years_to_check = sorted(set([latest_year, current_year]))

print(f"\nChecking Jolpica for rounds after {latest_year} R{latest_round}...")

def fetch_json(url, retries=MAX_RETRIES):
    """Fetch a Jolpica URL with 429-aware backoff."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=30)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                wait = RETRY_BACKOFF * (attempt + 1)
                print(f"  429 rate limit — waiting {wait:.0f}s...")
                time.sleep(wait)
            else:
                return None
        except Exception as e:
            print(f"  Error: {e}")
            time.sleep(RATE_SLEEP * 2)
    return None


# Build list of (year, round) tuples we need to fetch
rounds_needed = []

for year in years_to_check:
    data = fetch_json(f"{JOLPICA_BASE}/{year}?limit=100")
    time.sleep(RATE_SLEEP)
    if not data:
        continue
    try:
        race_list = data["MRData"]["RaceTable"]["Races"]
    except (KeyError, TypeError):
        continue

    for race in race_list:
        rnd = int(race["round"])
        # Skip rounds we already have
        if year == latest_year and rnd <= latest_round:
            continue
        # Only include rounds that have already occurred
        race_date = race.get("date", "")
        if race_date:
            try:
                rd = datetime.date.fromisoformat(race_date)
                if rd > datetime.date.today():
                    continue  # Future race — skip
            except ValueError:
                pass
        rounds_needed.append((year, rnd, race.get("raceName", f"Round {rnd}")))

if rounds_needed:
    print(f"\n{len(rounds_needed)} new round(s) to fetch:")
    for y, r, name in rounds_needed:
        print(f"  {y} Round {r:2d} — {name}")
else:
    print("\n✅ Data is already up to date — nothing to fetch.")

---
## Section 3: Fetch New Rounds from Jolpica

Fetch results, qualifying, sprint, and standings for each new round.
This cell is safe to re-run — it only appends rows for rounds not already present.

In [ ]:
if not rounds_needed:
    print("✅ Nothing to fetch — skipping.")
else:
    # ── Helper: build lookup dicts from existing tables ────────────────────
    def make_lookup(df, ref_col, id_col):
        return dict(zip(df[ref_col].astype(str), df[id_col]))

    driver_ref_to_id      = make_lookup(tables["drivers"],      "driver_ref",      "driver_id")
    constructor_ref_to_id = make_lookup(tables["constructors"],  "constructor_ref",  "constructor_id")
    circuit_ref_to_id     = make_lookup(tables["races"],         "circuit_id",       "circuit_id")  # placeholder

    # ID counters (pick up after existing maxes)
    next_ids = {
        "race_id":                  int(tables["races"]["race_id"].max()) + 1,
        "result_id":                int(tables["results"]["result_id"].max()) + 1,
        "qualify_id":               int(tables["qualifying"]["qualify_id"].max()) + 1,
        "sprint_result_id":         int(tables["sprint_results"]["sprint_result_id"].max()) + 1,
        "driver_standings_id":      int(tables["driver_standings"]["driver_standings_id"].max()) + 1,
        "constructor_standings_id": int(tables["constructor_standings"]["constructor_standings_id"].max()) + 1,
        "driver_id":                int(tables["drivers"]["driver_id"].max()) + 1,
        "constructor_id":           int(tables["constructors"]["constructor_id"].max()) + 1,
    }

    # Status string to ID
    # We don't load status table here, so use a simple incrementing approach
    # for any genuinely new status strings (rare)
    status_cache = {}  # status_str -> status_id (new ones only)

    def get_or_create_driver(driver_data):
        ref = driver_data.get("driverId", "")
        if ref in driver_ref_to_id:
            return driver_ref_to_id[ref]
        # New driver — add to drivers table
        did = next_ids["driver_id"]
        next_ids["driver_id"] += 1
        new_row = {
            "driver_id":   did,
            "driver_ref":  ref,
            "number":      pd.NA,
            "code":        driver_data.get("code", ""),
            "forename":    driver_data.get("givenName", ""),
            "surname":     driver_data.get("familyName", ""),
            "dob":         driver_data.get("dateOfBirth", ""),
            "nationality": driver_data.get("nationality", ""),
            "url":         driver_data.get("url", ""),
        }
        tables["drivers"] = pd.concat(
            [tables["drivers"], pd.DataFrame([new_row])], ignore_index=True
        )
        driver_ref_to_id[ref] = did
        print(f"    + New driver: {new_row['forename']} {new_row['surname']} ({ref})")
        return did

    def get_or_create_constructor(con_data):
        ref = con_data.get("constructorId", "")
        if ref in constructor_ref_to_id:
            return constructor_ref_to_id[ref]
        cid = next_ids["constructor_id"]
        next_ids["constructor_id"] += 1
        new_row = {
            "constructor_id":  cid,
            "constructor_ref": ref,
            "name":            con_data.get("name", ref),
            "nationality":     con_data.get("nationality", ""),
            "url":             con_data.get("url", ""),
        }
        tables["constructors"] = pd.concat(
            [tables["constructors"], pd.DataFrame([new_row])], ignore_index=True
        )
        constructor_ref_to_id[ref] = cid
        print(f"    + New constructor: {new_row['name']} ({ref})")
        return cid

    # ── Accumulator rows ───────────────────────────────────────────────────
    new_races = []
    new_results = []
    new_qualifying = []
    new_sprint = []
    new_drv_standings = []
    new_con_standings = []

    # ── Fetch loop ─────────────────────────────────────────────────────────
    for year, rnd, race_name in rounds_needed:
        print(f"\n  {year} Round {rnd:2d} — {race_name}")

        # ── Race entry ────────────────────────────────────────────────────
        race_data = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}?limit=1")
        time.sleep(RATE_SLEEP)

        race_id = next_ids["race_id"]
        if race_data:
            try:
                r = race_data["MRData"]["RaceTable"]["Races"][0]
                circuit_ref = r["Circuit"]["circuitId"]
                # Find circuit_id from existing races table
                circuit_match = tables["races"][
                    tables["races"]["circuit_id"].astype(str) == str(
                        tables["races"].loc[
                            tables["races"]["circuit_id"].astype(str).isin(
                                tables["races"]["circuit_id"].astype(str)
                            ), "circuit_id"
                        ].iloc[0] if not tables["races"].empty else pd.NA
                    )
                ]
                # Simpler: look up by joining with circuits if available
                # For now use a direct races lookup by circuit_ref via url pattern
                circuit_id_val = pd.NA

                new_races.append({
                    "race_id":    race_id,
                    "year":       year,
                    "round":      rnd,
                    "circuit_id": circuit_id_val,
                    "name":       r.get("raceName", race_name),
                    "date":       r.get("date", ""),
                    "time":       r.get("time", ""),
                    "url":        r.get("url", ""),
                })
                next_ids["race_id"] += 1
                print(f"    race_id={race_id}")
            except (KeyError, IndexError, TypeError) as e:
                print(f"    ⚠️  Race entry parse error: {e}")

        # ── Results ───────────────────────────────────────────────────────
        res_data = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/results?limit=30")
        time.sleep(RATE_SLEEP)

        if res_data:
            try:
                races_list = res_data["MRData"]["RaceTable"]["Races"]
                if races_list:
                    for entry in races_list[0].get("Results", []):
                        driver_id = get_or_create_driver(entry["Driver"])
                        constructor_id = get_or_create_constructor(entry["Constructor"])
                        new_results.append({
                            "result_id":         next_ids["result_id"],
                            "race_id":           race_id,
                            "driver_id":         driver_id,
                            "constructor_id":    constructor_id,
                            "number":            entry.get("number", pd.NA),
                            "grid":              int(entry.get("grid", 0)),
                            "position":          pd.to_numeric(entry.get("position"), errors="coerce"),
                            "position_text":     entry.get("positionText", ""),
                            "position_order":    int(entry.get("positionOrder", 0)),
                            "points":            float(entry.get("points", 0)),
                            "laps":              int(entry.get("laps", 0)),
                            "time":              entry.get("Time", {}).get("time", "") if "Time" in entry else "",
                            "milliseconds":      pd.to_numeric(entry.get("Time", {}).get("millis") if "Time" in entry else None, errors="coerce"),
                            "fastest_lap":       entry.get("FastestLap", {}).get("lap", pd.NA) if "FastestLap" in entry else pd.NA,
                            "rank":              pd.to_numeric(entry.get("FastestLap", {}).get("rank") if "FastestLap" in entry else None, errors="coerce"),
                            "fastest_lap_time":  entry.get("FastestLap", {}).get("Time", {}).get("time", "") if "FastestLap" in entry else "",
                            "fastest_lap_speed": pd.to_numeric(entry.get("FastestLap", {}).get("AverageSpeed", {}).get("speed") if "FastestLap" in entry else None, errors="coerce"),
                            "status_id":         1,  # placeholder; status lookup requires status table
                        })
                        next_ids["result_id"] += 1
                    print(f"    results: {len(races_list[0].get('Results', []))} entries")
            except (KeyError, IndexError, TypeError) as e:
                print(f"    ⚠️  Results parse error: {e}")
        time.sleep(RATE_SLEEP)

        # ── Qualifying ────────────────────────────────────────────────────
        qual_data = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/qualifying?limit=30")
        time.sleep(RATE_SLEEP)

        if qual_data:
            try:
                races_list = qual_data["MRData"]["RaceTable"]["Races"]
                if races_list:
                    for entry in races_list[0].get("QualifyingResults", []):
                        driver_id = get_or_create_driver(entry["Driver"])
                        constructor_id = get_or_create_constructor(entry["Constructor"])
                        new_qualifying.append({
                            "qualify_id":    next_ids["qualify_id"],
                            "race_id":       race_id,
                            "driver_id":     driver_id,
                            "constructor_id":constructor_id,
                            "number":        entry.get("number", pd.NA),
                            "position":      int(entry.get("position", 0)),
                            "q1":            entry.get("Q1", ""),
                            "q2":            entry.get("Q2", ""),
                            "q3":            entry.get("Q3", ""),
                        })
                        next_ids["qualify_id"] += 1
                    print(f"    qualifying: {len(races_list[0].get('QualifyingResults', []))} entries")
            except (KeyError, IndexError, TypeError) as e:
                print(f"    ⚠️  Qualifying parse error: {e}")

        # ── Sprint results ────────────────────────────────────────────────
        sprint_data = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/sprint?limit=30")
        time.sleep(RATE_SLEEP)

        if sprint_data:
            try:
                races_list = sprint_data["MRData"]["RaceTable"]["Races"]
                if races_list:
                    for entry in races_list[0].get("SprintResults", []):
                        driver_id = get_or_create_driver(entry["Driver"])
                        constructor_id = get_or_create_constructor(entry["Constructor"])
                        new_sprint.append({
                            "sprint_result_id": next_ids["sprint_result_id"],
                            "race_id":          race_id,
                            "driver_id":        driver_id,
                            "constructor_id":   constructor_id,
                            "number":           entry.get("number", pd.NA),
                            "grid":             int(entry.get("grid", 0)),
                            "position":         pd.to_numeric(entry.get("position"), errors="coerce"),
                            "position_text":    entry.get("positionText", ""),
                            "position_order":   int(entry.get("positionOrder", 0)),
                            "points":           float(entry.get("points", 0)),
                            "laps":             int(entry.get("laps", 0)),
                            "time":             entry.get("Time", {}).get("time", "") if "Time" in entry else "",
                            "milliseconds":     pd.to_numeric(entry.get("Time", {}).get("millis") if "Time" in entry else None, errors="coerce"),
                            "fastest_lap_time": entry.get("FastestLap", {}).get("Time", {}).get("time", "") if "FastestLap" in entry else "",
                            "status_id":        1,
                        })
                        next_ids["sprint_result_id"] += 1
                    if races_list[0].get("SprintResults"):
                        print(f"    sprint: {len(races_list[0]['SprintResults'])} entries")
            except (KeyError, IndexError, TypeError) as e:
                print(f"    ⚠️  Sprint parse error: {e}")

        # ── Driver standings ──────────────────────────────────────────────
        ds_data = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/driverStandings?limit=100")
        time.sleep(RATE_SLEEP)

        if ds_data:
            try:
                standings_lists = ds_data["MRData"]["StandingsTable"]["StandingsLists"]
                if standings_lists:
                    for entry in standings_lists[0].get("DriverStandings", []):
                        driver_id = get_or_create_driver(entry["Driver"])
                        new_drv_standings.append({
                            "driver_standings_id": next_ids["driver_standings_id"],
                            "race_id":             race_id,
                            "driver_id":           driver_id,
                            "points":              float(entry.get("points", 0)),
                            "position":            int(entry.get("position", 0)),
                            "position_text":       entry.get("positionText", ""),
                            "wins":                int(entry.get("wins", 0)),
                        })
                        next_ids["driver_standings_id"] += 1
                    print(f"    drv_standings: {len(standings_lists[0].get('DriverStandings', []))} entries")
            except (KeyError, IndexError, TypeError) as e:
                print(f"    ⚠️  Driver standings parse error: {e}")

        # ── Constructor standings ─────────────────────────────────────────
        cs_data = fetch_json(f"{JOLPICA_BASE}/{year}/{rnd}/constructorStandings?limit=100")
        time.sleep(RATE_SLEEP)

        if cs_data:
            try:
                standings_lists = cs_data["MRData"]["StandingsTable"]["StandingsLists"]
                if standings_lists:
                    for entry in standings_lists[0].get("ConstructorStandings", []):
                        constructor_id = get_or_create_constructor(entry["Constructor"])
                        new_con_standings.append({
                            "constructor_standings_id": next_ids["constructor_standings_id"],
                            "race_id":                  race_id,
                            "constructor_id":           constructor_id,
                            "points":                   float(entry.get("points", 0)),
                            "position":                 int(entry.get("position", 0)),
                            "position_text":            entry.get("positionText", ""),
                            "wins":                     int(entry.get("wins", 0)),
                        })
                        next_ids["constructor_standings_id"] += 1
                    print(f"    con_standings: {len(standings_lists[0].get('ConstructorStandings', []))} entries")
            except (KeyError, IndexError, TypeError) as e:
                print(f"    ⚠️  Constructor standings parse error: {e}")

    # ── Merge new rows into tables ─────────────────────────────────────────
    print("\n── Merging new rows ──────────────────────────────────────────────────")

    def merge_rows(table_name, new_rows, dedup_keys):
        if not new_rows:
            print(f"  {table_name:<30} no new rows")
            return
        new_df = pd.DataFrame(new_rows)
        combined = pd.concat([tables[table_name], new_df], ignore_index=True)
        # Dedup on natural key, keep last
        valid_keys = [k for k in dedup_keys if k in combined.columns]
        if valid_keys:
            combined = combined.drop_duplicates(subset=valid_keys, keep="last")
        tables[table_name] = combined.reset_index(drop=True)
        print(f"  {table_name:<30} +{len(new_rows)} rows → {len(tables[table_name]):,} total")

    merge_rows("races",                new_races,         ["year", "round"])
    merge_rows("results",              new_results,       ["race_id", "driver_id"])
    merge_rows("qualifying",           new_qualifying,    ["race_id", "driver_id"])
    merge_rows("sprint_results",       new_sprint,        ["race_id", "driver_id"])
    merge_rows("driver_standings",     new_drv_standings, ["race_id", "driver_id"])
    merge_rows("constructor_standings",new_con_standings, ["race_id", "constructor_id"])
    # drivers and constructors already updated in-place via get_or_create
    print(f"  {'drivers':<30} {len(tables['drivers']):,} total")
    print(f"  {'constructors':<30} {len(tables['constructors']):,} total")

    print("\n✅ All tables updated in memory")

---
## Section 4: Write Updated Parquet Files to GCS

Overwrites only the tables that were updated. Static tables are untouched.

In [ ]:
if not rounds_needed:
    print("✅ Data was already current — nothing to write.")
else:
    print(f"Writing updated Parquet files to gs://{GCS_BUCKET}/{GCS_BQ_STAGING}/...\n")

    bucket = gcs_client.bucket(GCS_BUCKET)

    for name in TABLES_TO_UPDATE:
        df = tables[name]
        # Write to an in-memory buffer and upload
        buf = io.BytesIO()
        df.to_parquet(buf, index=False)
        buf.seek(0)
        blob_path = f"{GCS_BQ_STAGING}/{name}.parquet"
        blob = bucket.blob(blob_path)
        blob.upload_from_file(buf, content_type="application/octet-stream")
        size_mb = buf.tell() / 1024 / 1024
        print(f"  ✅ {name:<30} {len(df):>8,} rows  ({size_mb:.2f} MB) → gs://{GCS_BUCKET}/{blob_path}")

    print("\n✅ All Parquet files updated in GCS")
    print("\n📋 Students loading from GCS in the lab will now get current data.")
    print("   Remember to rebuild the BQML model after loading if race prediction accuracy matters.")

---
## Section 5: Verification

In [ ]:
# Quick verification — show year-by-year race counts and latest round.

results_df   = tables["results"]
races_df     = tables["races"]

merged = results_df.merge(races_df[["race_id", "year", "round"]], on="race_id", how="left")

print("── Race coverage (last 5 seasons) ───────────────────────────────────")
recent = merged[merged["year"] >= merged["year"].max() - 4]
summary = recent.groupby("year").agg(
    rounds=("round", "nunique"),
    result_rows=("result_id", "count")
).reset_index().sort_values("year", ascending=False)

for _, row in summary.iterrows():
    print(f"  {int(row['year'])}: {int(row['rounds']):2d} rounds, {int(row['result_rows']):4d} result rows")

new_latest_year  = int(merged["year"].max())
new_latest_round = int(merged[merged["year"] == new_latest_year]["round"].max())

print(f"\n✅ Latest data now: {new_latest_year} Round {new_latest_round}")
print(f"   Total races table rows : {len(tables['races']):,}")
print(f"   Total results rows     : {len(tables['results']):,}")
print(f"   Total drivers          : {len(tables['drivers']):,}")
print(f"   Total constructors     : {len(tables['constructors']):,}")